# Obstacle Avoidance


Finally, we will use our object detection model to avoid duckie pedestrians as we navigate in Duckietown. 

The file that we will use for this [`model.py`](../../packages/solution/model.py).

The function that we need to write, with some helper functions, is `get_wheel_velocities_from_image`. You can expect that this will get called every time a new camera image comes in, and we should return wheel commands. The strategy we will take is very simple, if we detect a duckie pedestrian with sufficiently high confidence and sufficiently close, then we will stop (send wheel commands of 0), otherwise we will send fixed wheel command values to make the robot go (approximately in the real Duckiebot case) straight. 


The structure of this function is as follows:

```python
    def get_wheel_velocities_from_image(self, img: np.ndarray):
        try:
            detections = self._run_detector(img)
        except Exception as e:
            print(f"ONNX inference error {e}")
            return DifferentialPWM(left=0.0, right=0.0)
        if self._should_stop(detections):
            return DifferentialPWM(left=0.0, right=0.0)
        else:
            return DifferentialPWM(left=self.forward_pwm, right=self.forward_pwm)
```
Note that `FORWARD_PWM` is defined in your [solution/config.py](../../packages/solution/config.py). This will determine how fast your robot moves when there are (hopefully) no duckies in the way.

There are two function calls that we are making, the first is to `run_detector`. 
This function simply calls your ONNX model and returns the detections:

```python
    def _run_detector(self, img_bgr):
        x = self._preprocess(img_bgr)
        out = self.session.run(None, {self.input_name: x})[0]  # shape [1,N,6]
        return out[0]
```

Your job will be to process the detections and decide whether we should stop or not. For each detection if the following two conditions are met you should stop:
1. the confidence (score) is higher than the `CONF_THRESHOLD` defined in [solution/config.py](../../packages/solution/config.py).
2. the duckie is closer than `STOP_DISTANCE` away. You should figure out a way to estimate how far away it is. 

The code looks like this:

```python
    def _should_stop(self, detections: np.ndarray): 
        
        stop = False

        for x1, y1, x2, y2, score, _ in detections:


            # TODO we don't want to consider detections with confidence (score) below CONF_THRESHOLD (a value you should set in config.py)
            

            # TODO we want to stop if there is a duckie closer than STOP_DISTANCE away
            # To calculate if the duckie is too close we need to convert the pixel coordinates to 
            # world coordinates. To do so you can use the `self.ground_projector` object which has
            # loaded the camera extrinsic calibration
            # Specifically, if you want to project an object of type `pix = Pixel(x=u, y=v)` to a ground plane
            # point, you can first convert it to a vector (`vec = self.ground_projector.camera.pixel2vector(pix)`) and
            # then you can intersect that vector with the ground plane (`self.ground_projector.vector2ground(vec)`). 
            # That will be the point on the ground plane corresponding to the input pixel. 

        return stop
```

Once you are happy with your implementation you can test it. As described in the [`README`](../../README.md) you should first build

```
dts code build -R ROBOT_NAME [--local]
```

where you can add the `--local` flag to run the model on your laptop if you are using a real Duckiebot (not needed for virtual).

And then run with 

```
dts code workbench [-m] -R ROBOT_NAME [--local] 
```

where you should include `-m` if you are running a virtual robot in the Duckiematrix and you should include `--local` if you are running on a real Duckiebot. 

If all goes well your Duckiebot should STOP before any duckies get injured!

It can be useful to visualize the detections that are being made by your model. You can do so with the Image Viewer. Open it with 

```bash
dts duckiebot image_viewer ROBOT_NAME
```

In the top drop down you should be able to select a topic name `/ROBOT_NAME/object_detector_image/jpeg`. If you do so, that will show you an image with the bounding boxes overlayed as well as their associated confidences:

![image viewer object detection](../../assets/images/duckie_detection.png)
